# YOLO V8

# Subset: Components Only (PCB-MC-C)


In [2]:
!pip -q install -U ultralytics pyyaml

from ultralytics import YOLO
import yaml
from pathlib import Path

# -----------------------
# CONFIG
# -----------------------
KFOLD_PATH   = Path("/content/drive/MyDrive/PCB_MC/Data/components_only/kfold_data")
RESULTS_PATH = Path("/content/drive/MyDrive/PCB_MC/Results/YOLOV8/components_only")

RESULTS_PATH.mkdir(parents=True, exist_ok=True)

# Choose preset: "A" (recommended) or "B" (stronger)
PRESET = "A"

if PRESET == "A":
    BASE_WEIGHTS = "yolov8s.pt"
    IMGSZ = 1024
    BATCH = -1          # Ultralytics auto-batch (great on A100)
    EPOCHS = 300
elif PRESET == "B":
    BASE_WEIGHTS = "yolov8m.pt"
    IMGSZ = 1280
    BATCH = 8           # keep stable at higher res
    EPOCHS = 300
else:
    raise ValueError("PRESET must be 'A' or 'B'")

# Optimizer / schedule
OPTIMIZER = "AdamW"
LR0 = 1e-4
COS_LR = True
PATIENCE = 25

# Safe PCB augment (won't break footprints too much)
AUG = dict(
    fliplr=0.5,
    flipud=0.2,
    degrees=5.0,
    translate=0.05,
    scale=0.35,
    shear=0.0,
    perspective=0.0,
    hsv_h=0.015,
    hsv_s=0.40,
    hsv_v=0.30,
    mosaic=0.6,
    mixup=0.05,
    copy_paste=0.0
)

def load_and_validate_yaml(yaml_path: Path) -> dict:
    if not yaml_path.exists():
        raise FileNotFoundError(f"Missing YAML: {yaml_path}")

    with open(yaml_path, "r") as f:
        data = yaml.safe_load(f)

    if "names" not in data:
        raise ValueError(f"'names' not found in {yaml_path}. Make sure Roboflow YAML includes names.")
    if "nc" not in data:
        data["nc"] = len(data["names"])

    if data["nc"] != len(data["names"]):
        raise ValueError(f"Mismatch in {yaml_path}: nc={data['nc']} but len(names)={len(data['names'])}")

    if "val" not in data and "valid" in data:
        data["val"] = data["valid"]

    if "train" not in data or "val" not in data:
        raise ValueError(f"{yaml_path} must contain 'train' and 'val' (or 'valid'). Keys: {list(data.keys())}")

    return data

# -----------------------
# Train per fold
# -----------------------
for fold_num in range(5):
    fold_dir  = KFOLD_PATH / f"fold_{fold_num}"
    yaml_path = fold_dir / "data.yaml"

    cfg = load_and_validate_yaml(yaml_path)

    print(f"\n🚀 Training Fold {fold_num} | PRESET={PRESET}")
    print(f"  weights: {BASE_WEIGHTS} | imgsz: {IMGSZ} | batch: {BATCH}")
    print(f"  YAML: {yaml_path}")
    print(f"  nc: {cfg['nc']}")
    print(f"  names[0:5]: {cfg['names'][:5]} ... names[-3:]: {cfg['names'][-3:]}")

    model = YOLO(BASE_WEIGHTS)
    model.train(
        data=str(yaml_path),
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=BATCH,
        project=str(RESULTS_PATH),
        name=f"fold_{fold_num}",
        exist_ok=True,

        optimizer=OPTIMIZER,
        lr0=LR0,
        cos_lr=COS_LR,
        patience=PATIENCE,

        # Extra A100-friendly stability options
        amp=True,          # mixed precision (safe on A100)
        cache=True,        # cache images in RAM (often OK on Colab)
        workers=8,

        **AUG
    )

print("\n✅ Training complete for all folds!")


🚀 Training Fold 0 | PRESET=A
  weights: yolov8s.pt | imgsz: 1024 | batch: -1
  YAML: /content/drive/MyDrive/PCB_MC/Data/components_only/kfold_data/fold_0/data.yaml
  nc: 23
  names[0:5]: ['Button', 'Capacitor', 'Clock', 'Connector', 'Diode'] ... names[-3:]: ['Test Point', 'Transistor', 'Zener Diode']
Ultralytics 8.4.41 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA A100-SXM4-80GB, 81153MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/drive/MyDrive/PCB_MC/Data/components_only/kfold_data/fold_0/data.yaml, degrees=5.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=300, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.2, format=torchscript, fraction=1.0, freeze=No

# Subset: Full Dataset (PCB-MC-F)

In [1]:
!pip -q install -U ultralytics pyyaml

from ultralytics import YOLO
import yaml
from pathlib import Path

# -----------------------
# CONFIG
# -----------------------
KFOLD_PATH   = Path("/content/drive/MyDrive/PCB_MC/Data/full_dataset/kfold_data")
RESULTS_PATH = Path("/content/drive/MyDrive/PCB_MC/Results/YOLOV8/full_dataset")
RESULTS_PATH.mkdir(parents=True, exist_ok=True)

# Choose preset: "A" (recommended) or "B" (stronger)
PRESET = "A"

if PRESET == "A":
    BASE_WEIGHTS = "yolov8s.pt"
    IMGSZ = 1024
    BATCH = -1          # Ultralytics auto-batch (great on A100)
    EPOCHS = 300
elif PRESET == "B":
    BASE_WEIGHTS = "yolov8m.pt"
    IMGSZ = 1280
    BATCH = 8           # keep stable at higher res
    EPOCHS = 300
else:
    raise ValueError("PRESET must be 'A' or 'B'")

# Optimizer / schedule
OPTIMIZER = "AdamW"
LR0 = 1e-4
COS_LR = True
PATIENCE = 25

# Safe PCB augment (won't break footprints too much)
AUG = dict(
    fliplr=0.5,
    flipud=0.2,
    degrees=5.0,
    translate=0.05,
    scale=0.35,
    shear=0.0,
    perspective=0.0,
    hsv_h=0.015,
    hsv_s=0.40,
    hsv_v=0.30,
    mosaic=0.6,
    mixup=0.05,
    copy_paste=0.0
)

def load_and_validate_yaml(yaml_path: Path) -> dict:
    if not yaml_path.exists():
        raise FileNotFoundError(f"Missing YAML: {yaml_path}")

    with open(yaml_path, "r") as f:
        data = yaml.safe_load(f)

    if "names" not in data:
        raise ValueError(f"'names' not found in {yaml_path}. Make sure Roboflow YAML includes names.")
    if "nc" not in data:
        data["nc"] = len(data["names"])

    if data["nc"] != len(data["names"]):
        raise ValueError(f"Mismatch in {yaml_path}: nc={data['nc']} but len(names)={len(data['names'])}")

    if "val" not in data and "valid" in data:
        data["val"] = data["valid"]

    if "train" not in data or "val" not in data:
        raise ValueError(f"{yaml_path} must contain 'train' and 'val' (or 'valid'). Keys: {list(data.keys())}")

    return data

# -----------------------
# Train per fold
# -----------------------
for fold_num in range(5):
    fold_dir  = KFOLD_PATH / f"fold_{fold_num}"
    yaml_path = fold_dir / "data.yaml"

    cfg = load_and_validate_yaml(yaml_path)

    print(f"\n🚀 Training Fold {fold_num} | PRESET={PRESET}")
    print(f"  weights: {BASE_WEIGHTS} | imgsz: {IMGSZ} | batch: {BATCH}")
    print(f"  YAML: {yaml_path}")
    print(f"  nc: {cfg['nc']}")
    print(f"  names[0:5]: {cfg['names'][:5]} ... names[-3:]: {cfg['names'][-3:]}")

    model = YOLO(BASE_WEIGHTS)
    model.train(
        data=str(yaml_path),
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=BATCH,
        project=str(RESULTS_PATH),
        name=f"fold_{fold_num}",
        exist_ok=True,

        optimizer=OPTIMIZER,
        lr0=LR0,
        cos_lr=COS_LR,
        patience=PATIENCE,

        # Extra A100-friendly stability options
        amp=True,          # mixed precision (safe on A100)
        cache=True,        # cache images in RAM (often OK on Colab)
        workers=8,

        **AUG
    )

print("\n✅ Training complete for all folds!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 46.4 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.

🚀 Training Fold 0 | PRESET=A
  weights: yolov8s.pt | imgsz: 1024 | batch: -1
  YAML: /content/drive/MyDrive/PCB_MC/Data/full_dataset/kfold_data/fold_0/data.yaml
  nc: 31
  names[0:5]: ['Button', 'Capacitor', 'Clock', 'Connector', 'Diode'] ... names[-3:]: ['Test Point', 'Transistor', 'Zener Diode']
Ultralytics 8.4.41 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA A100-SXM4-80GB, 81153MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, c

# Subset: Missing Only (PCB-MC-M)

In [ ]:
!pip -q install -U ultralytics pyyaml

from ultralytics import YOLO
import yaml
from pathlib import Path

# -----------------------
# CONFIG
# -----------------------
KFOLD_PATH   = Path("/content/drive/MyDrive/PCB_MC/Data/missing_only/kfold_data")
RESULTS_PATH = Path("/content/drive/MyDrive/PCB_MC/Results/YOLOV8/missing_only")
RESULTS_PATH.mkdir(parents=True, exist_ok=True)

# Choose preset: "A" (recommended) or "B" (stronger)
PRESET = "A"

if PRESET == "A":
    BASE_WEIGHTS = "yolov8s.pt"
    IMGSZ = 1024
    BATCH = -1          # Ultralytics auto-batch (great on A100)
    EPOCHS = 300
elif PRESET == "B":
    BASE_WEIGHTS = "yolov8m.pt"
    IMGSZ = 1280
    BATCH = 8           # keep stable at higher res
    EPOCHS = 300
else:
    raise ValueError("PRESET must be 'A' or 'B'")

# Optimizer / schedule
OPTIMIZER = "AdamW"
LR0 = 1e-4
COS_LR = True
PATIENCE = 25

# Safe PCB augment (won't break footprints too much)
AUG = dict(
    fliplr=0.5,
    flipud=0.2,
    degrees=5.0,
    translate=0.05,
    scale=0.35,
    shear=0.0,
    perspective=0.0,
    hsv_h=0.015,
    hsv_s=0.40,
    hsv_v=0.30,
    mosaic=0.6,
    mixup=0.05,
    copy_paste=0.0
)

def load_and_validate_yaml(yaml_path: Path) -> dict:
    if not yaml_path.exists():
        raise FileNotFoundError(f"Missing YAML: {yaml_path}")

    with open(yaml_path, "r") as f:
        data = yaml.safe_load(f)

    if "names" not in data:
        raise ValueError(f"'names' not found in {yaml_path}. Make sure Roboflow YAML includes names.")
    if "nc" not in data:
        data["nc"] = len(data["names"])

    if data["nc"] != len(data["names"]):
        raise ValueError(f"Mismatch in {yaml_path}: nc={data['nc']} but len(names)={len(data['names'])}")

    if "val" not in data and "valid" in data:
        data["val"] = data["valid"]

    if "train" not in data or "val" not in data:
        raise ValueError(f"{yaml_path} must contain 'train' and 'val' (or 'valid'). Keys: {list(data.keys())}")

    return data

# -----------------------
# Train per fold
# -----------------------
for fold_num in range(5):
    fold_dir  = KFOLD_PATH / f"fold_{fold_num}"
    yaml_path = fold_dir / "data.yaml"

    cfg = load_and_validate_yaml(yaml_path)

    print(f"\n🚀 Training Fold {fold_num} | PRESET={PRESET}")
    print(f"  weights: {BASE_WEIGHTS} | imgsz: {IMGSZ} | batch: {BATCH}")
    print(f"  YAML: {yaml_path}")
    print(f"  nc: {cfg['nc']}")
    print(f"  names[0:5]: {cfg['names'][:5]} ... names[-3:]: {cfg['names'][-3:]}")

    model = YOLO(BASE_WEIGHTS)
    model.train(
        data=str(yaml_path),
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=BATCH,
        project=str(RESULTS_PATH),
        name=f"fold_{fold_num}",
        exist_ok=True,

        optimizer=OPTIMIZER,
        lr0=LR0,
        cos_lr=COS_LR,
        patience=PATIENCE,

        # Extra A100-friendly stability options
        amp=True,          # mixed precision (safe on A100)
        cache=True,        # cache images in RAM (often OK on Colab)
        workers=8,

        **AUG
    )

print("\n✅ Training complete for all folds!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 50.1 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.

🚀 Training Fold 0 | PRESET=A
  weights: yolov8s.pt | imgsz: 1024 | batch: -1
  YAML: /content/drive/MyDrive/PCB_MC/Data/missing_only/kfold_data/fold_0/data.yaml
  nc: 8
  names[0:5]: ['Missing Capacitor', 'Missing Component', 'Missing Diode', 'Missing Ferrite bead', 'Missing IC'] ... names[-3:]: ['Missing Inductor', 'Missing Led', 'Missing Resistor']
Ultralytics 8.4.34 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA A100-SXM4-80GB, 81153MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, 

# Subset: Non-missing (PCB-MC-P)

In [ ]:
!pip -q install -U ultralytics pyyaml

from ultralytics import YOLO
import yaml
from pathlib import Path

# -----------------------
# CONFIG
# -----------------------
KFOLD_PATH   = Path("/content/drive/MyDrive/PCB_MC/Data/non_missing/kfold_data")
RESULTS_PATH = Path("/content/drive/MyDrive/PCB_MC/Results/YOLOV8/non_missing")
RESULTS_PATH.mkdir(parents=True, exist_ok=True)

# Choose preset: "A" (recommended) or "B" (stronger)
PRESET = "A"

if PRESET == "A":
    BASE_WEIGHTS = "yolov8s.pt"
    IMGSZ = 1024
    BATCH = -1          # Ultralytics auto-batch (great on A100)
    EPOCHS = 300
elif PRESET == "B":
    BASE_WEIGHTS = "yolov8m.pt"
    IMGSZ = 1280
    BATCH = 8           # keep stable at higher res
    EPOCHS = 300
else:
    raise ValueError("PRESET must be 'A' or 'B'")

# Optimizer / schedule
OPTIMIZER = "AdamW"
LR0 = 1e-4
COS_LR = True
PATIENCE = 25

# Safe PCB augment (won't break footprints too much)
AUG = dict(
    fliplr=0.5,
    flipud=0.2,
    degrees=5.0,
    translate=0.05,
    scale=0.35,
    shear=0.0,
    perspective=0.0,
    hsv_h=0.015,
    hsv_s=0.40,
    hsv_v=0.30,
    mosaic=0.6,
    mixup=0.05,
    copy_paste=0.0
)

def load_and_validate_yaml(yaml_path: Path) -> dict:
    if not yaml_path.exists():
        raise FileNotFoundError(f"Missing YAML: {yaml_path}")

    with open(yaml_path, "r") as f:
        data = yaml.safe_load(f)

    if "names" not in data:
        raise ValueError(f"'names' not found in {yaml_path}. Make sure Roboflow YAML includes names.")
    if "nc" not in data:
        data["nc"] = len(data["names"])

    if data["nc"] != len(data["names"]):
        raise ValueError(f"Mismatch in {yaml_path}: nc={data['nc']} but len(names)={len(data['names'])}")

    if "val" not in data and "valid" in data:
        data["val"] = data["valid"]

    if "train" not in data or "val" not in data:
        raise ValueError(f"{yaml_path} must contain 'train' and 'val' (or 'valid'). Keys: {list(data.keys())}")

    return data

# -----------------------
# Train per fold
# -----------------------
for fold_num in range(5):
    fold_dir  = KFOLD_PATH / f"fold_{fold_num}"
    yaml_path = fold_dir / "data.yaml"

    cfg = load_and_validate_yaml(yaml_path)

    print(f"\n🚀 Training Fold {fold_num} | PRESET={PRESET}")
    print(f"  weights: {BASE_WEIGHTS} | imgsz: {IMGSZ} | batch: {BATCH}")
    print(f"  YAML: {yaml_path}")
    print(f"  nc: {cfg['nc']}")
    print(f"  names[0:5]: {cfg['names'][:5]} ... names[-3:]: {cfg['names'][-3:]}")

    model = YOLO(BASE_WEIGHTS)
    model.train(
        data=str(yaml_path),
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=BATCH,
        project=str(RESULTS_PATH),
        name=f"fold_{fold_num}",
        exist_ok=True,

        optimizer=OPTIMIZER,
        lr0=LR0,
        cos_lr=COS_LR,
        patience=PATIENCE,

        # Extra A100-friendly stability options
        amp=True,          # mixed precision (safe on A100)
        cache=True,        # cache images in RAM (often OK on Colab)
        workers=8,

        **AUG
    )

print("\n✅ Training complete for all folds!")


🚀 Training Fold 0 | PRESET=A
  weights: yolov8s.pt | imgsz: 1024 | batch: -1
  YAML: /content/drive/MyDrive/PCB_MC/Data/non_missing/kfold_data/fold_0/data.yaml
  nc: 23
  names[0:5]: ['Button', 'Capacitor', 'Clock', 'Connector', 'Diode'] ... names[-3:]: ['Test Point', 'Transistor', 'Zener Diode']
Ultralytics 8.4.34 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA A100-SXM4-80GB, 81153MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/drive/MyDrive/PCB_MC/Data/non_missing/kfold_data/fold_0/data.yaml, degrees=5.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=300, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.2, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_